# Partie F - Choix du modele IA et entrainement

## Objectif metier

L'objectif est de prevoir la demande touristique future par pays pour le prochain mois ou trimestre, afin d'aider l'equipe marketing a prioriser les pays et destinations a promouvoir.

La variable cible retenue est `target_next_period`, construite comme la demande du mois suivant par pays:

`target_next_period = demand_index shifted by -1 period per country`

Cette cible est une approximation gouvernee de la demande touristique future, car le fichier temporel disponible porte un `demand_index` mensuel par pays. Les volumes `visitors` sont disponibles au niveau pays-destination mais ne disposent pas d'un historique mensuel; ils sont donc utilises comme variables explicatives contextuelles et non comme cible temporelle directe.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import TimeSeriesSplit

from src.modeling import (
    prepare_modeling_dataset,
    temporal_train_test_split,
    train_full_model_suite,
)

processed = ROOT / "data" / "processed"
market = pd.read_csv(processed / "market_clean.csv", parse_dates=["month"])
gold = pd.read_csv(ROOT / "data" / "gold" / "gold_tourism_data.csv")

## 1. Type de probleme IA

Il s'agit d'une **regression supervisee** appliquee a une **prevision temporelle** sur des **donnees structurees**.

Le modele apprend a partir des observations historiques pour predire une demande future. Le caractere temporel impose un split chronologique afin d'eviter toute fuite de donnees.

Ce n'est pas une classification: la cible est une variable quantitative continue.

In [3]:
model_components = pd.DataFrame([
    ["Donnees d'entree X", "Pays, periode, demande courante, lags, moyenne mobile, visiteurs, attractivite, cout, note, sentiment, meteo, budget campagne"],
    ["Variable cible y", "target_next_period = demand_index du mois suivant par pays"],
    ["Algorithmes", "Persistence, moyenne mobile, SARIMAX, Prophet, Random Forest + lag features, XGBoost ou Gradient Boosting + lag features"],
    ["Fonction de perte", "Erreur de regression; optimisation indirecte via MAE/RMSE/MAPE"],
    ["Split temporel", "80 % premieres periodes en train, 20 % dernieres periodes en test"],
    ["Baselines", "Naive persistence et moving average 3 periodes"],
    ["Metriques", "MAE, RMSE, MAPE, R2"],
    ["Interpretation metier", "Mesurer la fiabilite d'une prediction utilisee pour prioriser les budgets marketing"],
    ["Limites", "Historique court, destination-level visitors non temporel, signaux sociaux potentiellement biaises"],
], columns=["Composante", "Choix et justification"])
model_components

,Composante,Choix et justification
0,Donnees d'entree X,"Pays, periode, demande courante, lags, moyenne..."
1,Variable cible y,target_next_period = demand_index du mois suiv...
2,Algorithmes,"Persistence, moyenne mobile, SARIMAX, Prophet,..."
3,Fonction de perte,Erreur de regression; optimisation indirecte v...
4,Split temporel,"80 % premieres periodes en train, 20 % dernier..."
5,Baselines,Naive persistence et moving average 3 periodes
6,Metriques,"MAE, RMSE, MAPE, R2"
7,Interpretation metier,Mesurer la fiabilite d'une prediction utilisee...
8,Limites,"Historique court, destination-level visitors n..."


## 2. Preparation des donnees temporelles

In [4]:
modeling_df = prepare_modeling_dataset(gold, market)
display(modeling_df.head())
display(modeling_df[[
    "country", "period", "visitors", "target_next_period",
    "attractiveness", "cost", "rating", "sentiment_score",
    "weather_score", "campaign_budget", "market_signal"
]].head())
modeling_df.isna().sum().to_frame("missing_values").T

,country,period,demand_index,visitors,attractiveness,cost,rating,sentiment_score,weather_score,campaign_budget,marketing_priority,market_signal,target_next_period,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_7,rolling_std_30,rolling_min_30,rolling_max_30,month_num,month,quarter,year,time_index
0,France,2022-01-01,91.800,185458668,8.730,859.300,4.289,-0.086,0.331,"120,000.000",0.295,91.800,97.800,108.300,108.500,110.400,112.200,110.900,118.500,109.500,111.600,114.964,115.267,10.828,12.005,89.600,128.700,1,1,1,2022,0
1,France,2022-02-01,97.800,185458668,8.730,859.300,4.289,-0.086,0.331,"120,000.000",0.295,97.800,92.500,91.800,108.500,110.400,112.200,110.900,118.500,91.800,91.800,91.800,91.800,10.828,12.005,91.800,91.800,2,2,1,2022,1
2,France,2022-03-01,92.500,185458668,8.730,859.300,4.289,-0.086,0.331,"120,000.000",0.295,92.500,92.500,97.800,91.800,110.400,112.200,110.900,118.500,94.800,94.800,94.800,94.800,4.243,4.243,91.800,97.800,3,3,1,2022,2
3,France,2022-04-01,92.500,185458668,8.730,859.300,4.289,-0.086,0.331,"120,000.000",0.295,92.500,90.000,92.500,97.800,91.800,112.200,110.900,118.500,94.033,94.033,94.033,94.033,3.281,3.281,91.800,97.800,4,4,2,2022,3
4,France,2022-05-01,90.000,185458668,8.730,859.300,4.289,-0.086,0.331,"120,000.000",0.295,90.000,87.400,92.500,92.500,97.800,112.200,110.900,118.500,94.267,93.650,93.650,93.650,2.786,2.786,91.800,97.800,5,5,2,2022,4


,country,period,visitors,target_next_period,attractiveness,cost,rating,sentiment_score,weather_score,campaign_budget,market_signal
0,France,2022-01-01,185458668,97.800,8.730,859.300,4.289,-0.086,0.331,"120,000.000",91.800
1,France,2022-02-01,185458668,92.500,8.730,859.300,4.289,-0.086,0.331,"120,000.000",97.800
2,France,2022-03-01,185458668,92.500,8.730,859.300,4.289,-0.086,0.331,"120,000.000",92.500
3,France,2022-04-01,185458668,90.000,8.730,859.300,4.289,-0.086,0.331,"120,000.000",92.500
4,France,2022-05-01,185458668,87.400,8.730,859.300,4.289,-0.086,0.331,"120,000.000",90.000


,country,period,demand_index,visitors,attractiveness,cost,rating,sentiment_score,weather_score,campaign_budget,marketing_priority,market_signal,target_next_period,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_7,rolling_std_30,rolling_min_30,rolling_max_30,month_num,month,quarter,year,time_index
missing_values,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Interpretation : la colonne `period` existe dans les signaux marche et est convertie en datetime. La table de modelisation est triee par pays et periode. Les lignes sans `target_next_period` sont supprimees uniquement parce qu'elles correspondent a la derniere periode observee, pour laquelle le futur n'est pas encore connu.

## 2.1 Feature engineering temporel sans fuite de donnees

In [5]:
time_features = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_30",
    "rolling_mean_3", "rolling_mean_7", "rolling_mean_14", "rolling_mean_30",
    "rolling_std_7", "rolling_std_30", "rolling_min_30", "rolling_max_30",
    "month", "quarter", "year",
]
modeling_df[["country", "period", "market_signal", "target_next_period"] + time_features].head(12)

,country,period,market_signal,target_next_period,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_7,rolling_std_30,rolling_min_30,rolling_max_30,month,quarter,year
0,France,2022-01-01,91.800,97.800,108.300,108.500,110.400,112.200,110.900,118.500,109.500,111.600,114.964,115.267,10.828,12.005,89.600,128.700,1,1,2022
1,France,2022-02-01,97.800,92.500,91.800,108.500,110.400,112.200,110.900,118.500,91.800,91.800,91.800,91.800,10.828,12.005,91.800,91.800,2,1,2022
2,France,2022-03-01,92.500,92.500,97.800,91.800,110.400,112.200,110.900,118.500,94.800,94.800,94.800,94.800,4.243,4.243,91.800,97.800,3,1,2022
3,France,2022-04-01,92.500,90.000,92.500,97.800,91.800,112.200,110.900,118.500,94.033,94.033,94.033,94.033,3.281,3.281,91.800,97.800,4,2,2022
4,France,2022-05-01,90.000,87.400,92.500,92.500,97.800,112.200,110.900,118.500,94.267,93.650,93.650,93.650,2.786,2.786,91.800,97.800,5,2,2022
5,France,2022-06-01,87.400,88.400,90.000,92.500,92.500,112.200,110.900,118.500,91.667,92.920,92.920,92.920,2.913,2.913,90.000,97.800,6,2,2022
6,France,2022-07-01,88.400,62.300,87.400,90.000,92.500,112.200,110.900,118.500,89.967,92.000,92.000,92.000,3.445,3.445,87.400,97.800,7,3,2022
7,France,2022-08-01,62.300,67.900,88.400,87.400,90.000,91.800,110.900,118.500,88.600,91.486,91.486,91.486,3.427,3.427,87.400,97.800,8,3,2022
8,France,2022-09-01,67.900,77.700,62.300,88.400,87.400,97.800,110.900,118.500,79.367,87.271,87.838,87.838,11.531,10.795,62.300,97.800,9,3,2022
9,France,2022-10-01,77.700,89.000,67.900,62.300,88.400,92.500,110.900,118.500,72.867,83.000,85.622,85.622,12.480,12.089,62.300,97.800,10,4,2022


Toutes les variables glissantes utilisent `shift(1)` avant le calcul des moyennes, ecarts-types, minimums et maximums. Cela garantit qu'une observation future ou courante ne fuit pas dans les variables explicatives servant a predire la periode suivante.

## 2.2 Validation croisee temporelle avec TimeSeriesSplit

In [6]:
tscv = TimeSeriesSplit(n_splits=5, test_size=None)
ordered_for_cv = modeling_df.sort_values(["period", "country"]).reset_index(drop=True)

fold_rows = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(ordered_for_cv), start=1):
    fold_rows.append({
        "Fold": fold,
        "Train start": ordered_for_cv.loc[train_idx, "period"].min(),
        "Train end": ordered_for_cv.loc[train_idx, "period"].max(),
        "Test start": ordered_for_cv.loc[test_idx, "period"].min(),
        "Test end": ordered_for_cv.loc[test_idx, "period"].max(),
        "Train rows": len(train_idx),
        "Test rows": len(test_idx),
    })
folds = pd.DataFrame(fold_rows)
display(folds)

fig = go.Figure()
for _, row in folds.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["Train start"], row["Train end"]],
        y=[f"Fold {row['Fold']}", f"Fold {row['Fold']}"],
        mode="lines",
        line=dict(color="#2563EB", width=10),
        name="Train" if row["Fold"] == 1 else None,
        showlegend=row["Fold"] == 1,
    ))
    fig.add_trace(go.Scatter(
        x=[row["Test start"], row["Test end"]],
        y=[f"Fold {row['Fold']}", f"Fold {row['Fold']}"],
        mode="lines",
        line=dict(color="#F59E0B", width=10),
        name="Test" if row["Fold"] == 1 else None,
        showlegend=row["Fold"] == 1,
    ))
fig.update_layout(template="plotly_white", title="Visualisation des folds TimeSeriesSplit", xaxis_title="Periode", yaxis_title="")
fig.show()

,Fold,Train start,Train end,Test start,Test end,Train rows,Test rows
0,1,2022-01-01,2022-07-01,2022-07-01,2022-12-01,49,46
1,2,2022-01-01,2022-12-01,2023-01-01,2023-06-01,95,46
2,3,2022-01-01,2023-06-01,2023-06-01,2023-12-01,141,46
3,4,2022-01-01,2023-12-01,2023-12-01,2024-06-01,187,46
4,5,2022-01-01,2024-06-01,2024-06-01,2024-11-01,233,46


TimeSeriesSplit respecte l'ordre chronologique des observations et permet de valider le modele sur plusieurs fenetres temporelles successives sans fuite d'information.

Cette validation est appliquee aux modeles de Machine Learning avec lag features : Random Forest et XGBoost/Gradient Boosting. Les modeles statistiques comme SARIMAX sont generalement evalues sur un decoupage temporel chronologique ou via une approche rolling forecast plutot qu'avec une validation croisee standard. Prophet est conserve sur une evaluation hold-out; si la librairie est disponible, ses outils de diagnostics peuvent etre ajoutes via `prophet.diagnostics.cross_validation`.

## 3. Split temporel

In [7]:
train, test = temporal_train_test_split(modeling_df, test_ratio=0.2)
split_summary = pd.DataFrame({
    "train_start": [train["period"].min()],
    "train_end": [train["period"].max()],
    "test_start": [test["period"].min()],
    "test_end": [test["period"].max()],
    "train_rows": [len(train)],
    "test_rows": [len(test)],
    "countries_train": [train["country"].nunique()],
    "countries_test": [test["country"].nunique()],
})
split_summary

,train_start,train_end,test_start,test_end,train_rows,test_rows,countries_train,countries_test
0,2022-01-01,2024-04-01,2024-05-01,2024-11-01,223,56,8,8


Le split temporel respecte l'ordre chronologique des observations et simule une situation reelle ou le futur n'est pas connu au moment de l'entrainement. Un `train_test_split` aleatoire serait une fuite de donnees et surestimerait la performance.

## 4. Baselines et modeles avances

In [8]:
suite = train_full_model_suite(
    gold=gold,
    market=market,
    reports_dir=ROOT / "reports",
    gold_dir=ROOT / "data" / "gold",
    models_dir=ROOT / "models",
)

performance = suite["performance"]
predictions = suite["predictions"]
feature_importance = suite["feature_importance"]
cv_results = suite["cv_results"]
cv_summary = suite["cv_summary"]
limitations = suite["limitations"]

performance

,Model,Approach type,MAE,RMSE,MAPE,R2,Business interpretation,Status
5,Gradient Boosting + lag features,Supervised forecasting with lag features,6.853,8.674,7.302,0.856,Candidate model for country demand forecasting...,OK
4,Random Forest + lag features,Supervised forecasting with lag features,7.690,9.640,8.150,0.822,Candidate model for country demand forecasting...,OK
0,Baseline Persistence,Temporal baseline,9.939,12.156,10.701,0.717,Reference point used to prove whether advanced...,OK
1,Baseline Moving Average,Temporal baseline,13.237,14.941,14.311,0.572,Reference point used to prove whether advanced...,OK
2,SARIMAX,Statistical time-series model,16.886,20.391,19.198,0.203,Candidate model for country demand forecasting...,OK
3,Prophet,Additive time-series model,NaN,NaN,NaN,NaN,Model unavailable or no valid predictions.,Unavailable


Modeles entraines :

- **Baseline Persistence** : prevision(t+1) = valeur observee a t.
- **Baseline Moving Average** : prevision(t+1) = moyenne mobile des 3 dernieres periodes.
- **SARIMAX** : modele statistique de serie temporelle entraine par pays avec variables exogenes et `try/except`.
- **Prophet** : modele additif de serie temporelle par pays, avec regressors si disponible et fallback si la librairie n'est pas installee.
- **RandomForestRegressor + lag features** : pipeline sklearn avec `ColumnTransformer`, `OneHotEncoder`, imputation, `StandardScaler`, lags et rolling features.
- **XGBoost ou Gradient Boosting + lag features** : XGBoost est utilise si installe, sinon fallback propre vers `GradientBoostingRegressor`, toujours avec variables temporelles.

Les metriques suivies sont `MAE`, `RMSE`, `MAPE` et `R2`. Le R2 mesure la part de variance expliquee par le modele; il peut etre negatif si le modele fait moins bien qu'une prediction moyenne naive.

## 5.1 Resultats TimeSeriesSplit par fold

In [9]:
display(cv_results)
display(cv_summary)

,Model,Fold,Train start,Train end,Test start,Test end,MAE,RMSE,MAPE
0,Random Forest + lag features,1,2022-01-01,2022-07-01,2022-07-01,2022-12-01,10.907,13.923,10.613
1,Gradient Boosting + lag features,1,2022-01-01,2022-07-01,2022-07-01,2022-12-01,16.308,20.119,16.211
2,Random Forest + lag features,2,2022-01-01,2022-12-01,2023-01-01,2023-06-01,10.511,12.796,9.842
3,Gradient Boosting + lag features,2,2022-01-01,2022-12-01,2023-01-01,2023-06-01,10.672,12.247,9.734
4,Random Forest + lag features,3,2022-01-01,2023-06-01,2023-06-01,2023-12-01,9.504,11.197,10.176
5,Gradient Boosting + lag features,3,2022-01-01,2023-06-01,2023-06-01,2023-12-01,8.657,10.809,8.977
6,Random Forest + lag features,4,2022-01-01,2023-12-01,2023-12-01,2024-06-01,5.461,6.903,4.663
7,Gradient Boosting + lag features,4,2022-01-01,2023-12-01,2023-12-01,2024-06-01,5.256,6.909,4.709
8,Random Forest + lag features,5,2022-01-01,2024-06-01,2024-06-01,2024-11-01,7.717,9.609,8.432
9,Gradient Boosting + lag features,5,2022-01-01,2024-06-01,2024-06-01,2024-11-01,6.723,8.687,7.317


,Model,Mean_MAE,Std_MAE,Mean_RMSE,Std_RMSE,Mean_MAPE,Std_MAPE
1,Random Forest + lag features,8.820,2.246,10.886,2.760,8.746,2.424
0,Gradient Boosting + lag features,9.523,4.306,11.754,5.099,9.390,4.273


## 5.2 Visualisations de robustesse TimeSeriesSplit

In [10]:
fig = px.box(
    cv_results,
    x="Model",
    y="RMSE",
    color="Model",
    color_discrete_sequence=["#2563EB", "#14B8A6"],
    title="Boxplot RMSE par fold",
)
fig.update_layout(template="plotly_white", xaxis_title="", yaxis_title="RMSE", showlegend=False)
fig.show()

fig = px.box(
    cv_results,
    x="Model",
    y="MAE",
    color="Model",
    color_discrete_sequence=["#2563EB", "#14B8A6"],
    title="Boxplot MAE par fold",
)
fig.update_layout(template="plotly_white", xaxis_title="", yaxis_title="MAE", showlegend=False)
fig.show()

cv_long = cv_results.melt(
    id_vars=["Model", "Fold"],
    value_vars=["MAE", "RMSE", "MAPE"],
    var_name="Metric",
    value_name="Value",
)
fig = px.line(
    cv_long,
    x="Fold",
    y="Value",
    color="Model",
    line_dash="Metric",
    markers=True,
    title="Evolution des performances selon les folds",
)
fig.update_layout(template="plotly_white", xaxis_title="Fold chronologique", yaxis_title="Erreur")
fig.show()

Interpretation : la validation croisee temporelle mesure la stabilite des modeles ML sur plusieurs fenetres successives. Une faible dispersion des RMSE/MAE indique une meilleure robustesse temporelle. Cette verification complete le hold-out final et reduit le risque de selectionner un modele performant uniquement sur une seule fenetre de test.

## 5. Tableau comparatif des performances

In [11]:
display(performance)
if not limitations.empty:
    display(limitations)

,Model,Approach type,MAE,RMSE,MAPE,R2,Business interpretation,Status
5,Gradient Boosting + lag features,Supervised forecasting with lag features,6.853,8.674,7.302,0.856,Candidate model for country demand forecasting...,OK
4,Random Forest + lag features,Supervised forecasting with lag features,7.690,9.640,8.150,0.822,Candidate model for country demand forecasting...,OK
0,Baseline Persistence,Temporal baseline,9.939,12.156,10.701,0.717,Reference point used to prove whether advanced...,OK
1,Baseline Moving Average,Temporal baseline,13.237,14.941,14.311,0.572,Reference point used to prove whether advanced...,OK
2,SARIMAX,Statistical time-series model,16.886,20.391,19.198,0.203,Candidate model for country demand forecasting...,OK
3,Prophet,Additive time-series model,NaN,NaN,NaN,NaN,Model unavailable or no valid predictions.,Unavailable


,model,country,issue
0,Prophet,ALL,prophet is not installed; Prophet skipped grac...


## 6. Graphique de comparaison MAE / RMSE / MAPE

In [12]:
fig = px.bar(
    performance.dropna(subset=["RMSE"]).sort_values("RMSE", ascending=False),
    x="RMSE",
    y="Model",
    color="Approach type",
    orientation="h",
    color_discrete_sequence=["#2563EB", "#14B8A6", "#F59E0B", "#64748B"],
    title="Comparaison des modeles par RMSE",
)
fig.update_layout(template="plotly_white", xaxis_title="RMSE", yaxis_title="")
fig.show()

fig = px.bar(
    performance.dropna(subset=["MAPE"]).sort_values("MAPE", ascending=False),
    x="MAPE",
    y="Model",
    color="Approach type",
    orientation="h",
    color_discrete_sequence=["#2563EB", "#14B8A6", "#F59E0B", "#64748B"],
    title="Comparaison des modeles par MAPE",
)
fig.update_layout(template="plotly_white", xaxis_title="MAPE (%)", yaxis_title="")
fig.show()

Interpretation : les baselines sont le point de comparaison minimal. Un modele avance doit les battre pour justifier sa complexite. Le RMSE penalise fortement les grosses erreurs, ce qui est pertinent pour une decision budgetaire.

## 7. Reel vs predit pour le meilleur modele

In [13]:
model_column_map = {
    "Baseline Persistence": "baseline_persistence",
    "Baseline Moving Average": "baseline_moving_average",
    "Random Forest + lag features": "random_forest",
    "XGBoost + lag features": "gradient_boosting",
    "Gradient Boosting + lag features": "gradient_boosting",
    "SARIMAX": "sarimax",
    "Prophet": "prophet",
}
best_model = performance[performance["Status"] == "OK"].iloc[0]["Model"]
best_column = model_column_map[best_model]
sample_country = predictions["country"].iloc[0]
country_plot = predictions[predictions["country"] == sample_country].sort_values("period")

fig = go.Figure()
fig.add_trace(go.Scatter(x=country_plot["period"], y=country_plot["actual"], mode="lines+markers", name="Reel", line=dict(color="#1E293B", width=3)))
fig.add_trace(go.Scatter(x=country_plot["period"], y=country_plot[best_column], mode="lines+markers", name=best_model, line=dict(color="#2563EB", width=3)))
fig.update_layout(template="plotly_white", title=f"Reel vs predit - {sample_country}", xaxis_title="Periode", yaxis_title="Demande future")
fig.show()

Interpretation : cette courbe permet de verifier si le modele suit correctement la dynamique temporelle sur les periodes de test, et pas uniquement une moyenne globale.

## 8. Scatter plot y_true vs y_pred et residus

In [14]:
eval_df = predictions[["actual", best_column]].dropna().rename(columns={best_column: "prediction"})
eval_df["residual"] = eval_df["actual"] - eval_df["prediction"]

fig = px.scatter(
    eval_df,
    x="actual",
    y="prediction",
    color="residual",
    color_continuous_scale="RdBu",
    title=f"Actual vs predicted - {best_model}",
)
fig.add_trace(go.Scatter(x=eval_df["actual"], y=eval_df["actual"], mode="lines", name="Ideal", line=dict(color="#1E293B", dash="dash")))
fig.update_layout(template="plotly_white", xaxis_title="Valeur reelle", yaxis_title="Prediction")
fig.show()

fig = px.histogram(
    eval_df,
    x="residual",
    nbins=20,
    color_discrete_sequence=["#2563EB"],
    title=f"Distribution des residus - {best_model}",
)
fig.update_layout(template="plotly_white", xaxis_title="Erreur reelle - predite", yaxis_title="Frequence")
fig.show()

Interpretation : un bon modele doit produire des points proches de la diagonale et des residus centres autour de zero. Des residus asymetriques indiqueraient un biais de sur- ou sous-prediction.

## 9. Feature importance Random Forest

In [15]:
display(feature_importance.head(15))
fig = px.bar(
    feature_importance.head(15).sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    color_discrete_sequence=["#14B8A6"],
    title="Top feature importance - Random Forest",
)
fig.update_layout(template="plotly_white", xaxis_title="Importance", yaxis_title="")
fig.show()

,feature,importance
0,market_signal,0.807
1,rolling_std_30,0.031
2,month_num,0.020
3,month,0.018
4,rolling_mean_14,0.012
5,lag_1,0.012
6,lag_7,0.011
7,rolling_std_7,0.009
8,rolling_mean_30,0.009
9,lag_3,0.009


Interpretation : l'importance des variables aide a expliquer les drivers de prediction. Elle doit etre lue comme une indication de contribution statistique, pas comme une causalite metier.

## 10. Choix du meilleur modele et lecture critique

In [16]:
best_row = performance[performance["Status"] == "OK"].iloc[0]
generated_conclusion = (
    f"Le modele retenu est {best_row['Model']} car il obtient la meilleure performance "
    f"sur l'ensemble de test avec un RMSE de {best_row['RMSE']:.2f}, un MAPE de {best_row['MAPE']:.2f}% "
    f"et un R2 de {best_row['R2']:.2f}. "
    "Il sera utilise pour alimenter la GOLD DATA et le moteur de recommandation, sous reserve de validation metier."
)
print(generated_conclusion)

Le modele retenu est Gradient Boosting + lag features car il obtient la meilleure performance sur l'ensemble de test avec un RMSE de 8.67, un MAPE de 7.30% et un R2 de 0.86. Il sera utilise pour alimenter la GOLD DATA et le moteur de recommandation, sous reserve de validation metier.


Lecture critique :

- La meilleure performance statistique ne suffit pas : il faut verifier la robustesse dans le temps.
- Random Forest et Gradient Boosting ne sont acceptables ici que parce qu'ils utilisent des lags, rolling features et variables calendaires.
- SARIMAX et Prophet sont methodologiquement pertinents, mais limites par la longueur d'historique disponible.
- L'interpretabilite doit etre renforcee par feature importance, analyse des residus et validation metier.
- Les donnees sociales, meteo et campagnes contiennent des biais et incoherences documentes dans la gouvernance.

Formulation de synthese : les modeles Random Forest et XGBoost ont ete adaptes au forecasting via la creation de variables de retard et de moyennes glissantes. Ils ne sont donc pas utilises comme simples modeles tabulaires, mais comme approches de prevision supervisee temporelle.

## 11. Sorties generees

In [17]:
outputs = [
    ROOT / "reports" / "model_performance.csv",
    ROOT / "reports" / "forecast_predictions.csv",
    ROOT / "reports" / "time_series_cv_results.csv",
    ROOT / "reports" / "time_series_cv_summary.csv",
    ROOT / "data" / "gold" / "forecast_predictions.csv",
    ROOT / "data" / "gold" / "gold_tourism_data_with_forecast.csv",
    ROOT / "reports" / "best_model_summary.md",
    ROOT / "models" / "best_model.pkl",
    ROOT / "reports" / "feature_importance.csv",
    ROOT / "reports" / "time_series_model_limits.csv",
]
pd.DataFrame({"output": [str(path) for path in outputs], "exists": [path.exists() for path in outputs]})

,output,exists
0,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
1,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
2,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
3,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
4,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
5,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
6,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
7,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
8,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True
9,C:\Users\bahae\Downloads\epitaexamenfinalconsi...,True


Recommandations d'amelioration :

- Collecter un historique mensuel de visiteurs par destination, pas seulement par pays.
- Ajouter des variables calendaires: vacances scolaires, evenements, saisons, prix moyens.
- Monitorer le drift des donnees et la degradation des metriques.
- Mettre en place un backtesting multi-fenetres.
- Valider les recommandations par tests A/B marketing.